## 파이썬 데이터 그룹 연산과 집계 파이프라인


---
## 1. 환경 설정 및 라이브러리 임포트

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

# 출력 옵션 설정
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print(f"Pandas 버전: {pd.__version__}")
print(f"Seaborn 버전: {sns.__version__}")

Pandas 버전: 2.2.2
Seaborn 버전: 0.13.2


---
## 2. 분석 대상 데이터셋: Titanic

seaborn 라이브러리의 titanic 데이터셋을 활용합니다.

**분석 대상 열 추출:**
- `survived`: 생존 여부 (0=사망, 1=생존)
- `sex`: 성별
- `age`: 나이
- `fare`: 요금
- `embark_town`: 승선 항구

**목표:** 생존 여부(survived) 및 성별(sex) 기준의 다중 그룹 연산 파이프라인 구축

In [2]:
# Titanic 데이터셋 로드
titan = sns.load_dataset('titanic')

# 분석 대상 열 추출
df = titan.loc[:, ['survived', 'sex', 'age', 'embark_town']]

print("=" * 60)
print("데이터 기본 정보")
print("=" * 60)
print(f"\n데이터 크기: {df.shape}")
print(f"\n컬럼 목록: {list(df.columns)}")
print(f"\n데이터 타입:\n{df.dtypes}")
print(f"\n결측값:\n{df.isnull().sum()}")
df.head(10)

데이터 기본 정보

데이터 크기: (891, 4)

컬럼 목록: ['survived', 'sex', 'age', 'embark_town']

데이터 타입:
survived         int64
sex             object
age            float64
embark_town     object
dtype: object

결측값:
survived         0
sex              0
age            177
embark_town      2
dtype: int64


,survived,sex,age,embark_town
0,0,male,22.0,Southampton
1,1,female,38.0,Cherbourg
2,1,female,26.0,Southampton
3,1,female,35.0,Southampton
4,0,male,35.0,Southampton
5,0,male,NaN,Queenstown
6,0,male,54.0,Southampton
7,0,male,2.0,Southampton
8,1,female,27.0,Southampton
9,1,female,14.0,Cherbourg


---
## 4. Split-Apply-Combine 3단계

| 단계 | 영문 | 설명 |
|------|------|------|
| **Phase 1** | Split (분할) | 데이터를 특정 조건(Key)에 의해 독립된 그룹으로 분할 |
| **Phase 2** | Apply (집계 및 변환) | 그룹별 독립적 연산 처리 (평균, 합계 등) |
| **Phase 3** | Combine (결합) | Apply 단계의 개별 연산 결과를 하나의 데이터 구조로 병합 |

---
## 5. Phase 1: Split (분할) - 그룹 객체 생성

### 5.1 groupby() 함수

In [3]:
# 단일 기준 그룹화: 'survived' 기준
group_survived = df.groupby('survived')
print("단일 기준 그룹화 (survived):")
print(type(group_survived))
print(f"그룹 수: {group_survived.ngroups}")
print(f"그룹 키: {list(group_survived.groups.keys())}")

단일 기준 그룹화 (survived):
<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
그룹 수: 2
그룹 키: [0, 1]


In [4]:
# 다중 기준 그룹화: 'survived'와 'sex' 기준
group_res = df.groupby(['survived', 'sex'])
print("다중 기준 그룹화 (survived, sex):")
print(type(group_res))
print(f"그룹 수: {group_res.ngroups}")
print(f"\n그룹 키 목록:")
for key in group_res.groups.keys():
    print(f"  {key}: {len(group_res.groups[key])}건")

다중 기준 그룹화 (survived, sex):
<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
그룹 수: 4

그룹 키 목록:
  (0, 'female'): 81건
  (0, 'male'): 468건
  (1, 'female'): 233건
  (1, 'male'): 109건


### 5.2 DataFrameGroupBy 객체

In [5]:
# DataFrameGroupBy 객체 확인
print("DataFrameGroupBy 객체 정보:")
print(f"타입: {type(group_res)}")
print(f"\n객체 출력 (연산 결과가 아닌 객체 참조):")
print(group_res)

DataFrameGroupBy 객체 정보:
타입: <class 'pandas.core.groupby.generic.DataFrameGroupBy'>

객체 출력 (연산 결과가 아닌 객체 참조):


In [6]:
# 각 그룹의 내용 확인 (get_group)
print("=" * 60)
print("그룹별 데이터 확인: survived=0, sex='male' (사망 남성)")
print("=" * 60)
group_res.get_group((0, 'male')).head()

그룹별 데이터 확인: survived=0, sex='male' (사망 남성)


,survived,sex,age,embark_town
0,0,male,22.0,Southampton
4,0,male,35.0,Southampton
5,0,male,NaN,Queenstown
6,0,male,54.0,Southampton
7,0,male,2.0,Southampton


In [7]:
# 그룹 순회 (iteration)
print("=" * 60)
print("그룹 순회 - 각 그룹의 크기 확인")
print("=" * 60)
for name, group in group_res:
    print(f"그룹 {name}: {len(group)}건, 평균 나이: {group['age'].mean():.2f}")

그룹 순회 - 각 그룹의 크기 확인
그룹 (np.int64(0), 'female'): 81건, 평균 나이: 25.05
그룹 (np.int64(0), 'male'): 468건, 평균 나이: 31.62
그룹 (np.int64(1), 'female'): 233건, 평균 나이: 28.85
그룹 (np.int64(1), 'male'): 109건, 평균 나이: 27.28


---
## 6. Phase 2: Apply (집계 및 변환) - 그룹 객체 연산

### 6.1 기본 집계 함수

| 함수 | 설명 |
|------|------|
| `mean()` | 평균 |
| `max()` | 최대값 |
| `min()` | 최소값 |
| `sum()` | 합계 |
| `count()` | 데이터 개수 |
| `size()` | 그룹 크기 |
| `var()` | 분산 |
| `std()` | 표준편차 |

In [8]:
# mean() - 평균
print("=" * 60)
print("그룹별 평균 (survived, sex 기준)")
print("=" * 60)
print(group_res.mean(numeric_only=True))

그룹별 평균 (survived, sex 기준)
                       age
survived sex              
0        female  25.046875
         male    31.618056
1        female  28.847716
         male    27.276022


In [9]:
# max(), min() - 최대값, 최소값
print("=" * 60)
print("그룹별 최대값")
print("=" * 60)
print(group_res.max(numeric_only=True))

print("\n" + "=" * 60)
print("그룹별 최소값")
print("=" * 60)
print(group_res.min(numeric_only=True))

그룹별 최대값
                  age
survived sex         
0        female  57.0
         male    74.0
1        female  63.0
         male    80.0

그룹별 최소값
                  age
survived sex         
0        female  2.00
         male    1.00
1        female  0.75
         male    0.42


In [10]:
# sum(), count(), size()
print("=" * 60)
print("그룹별 합계")
print("=" * 60)
print(group_res.sum(numeric_only=True))

print("\n" + "=" * 60)
print("그룹별 데이터 개수 (count - 결측값 제외)")
print("=" * 60)
print(group_res.count())

print("\n" + "=" * 60)
print("그룹별 크기 (size - 결측값 포함)")
print("=" * 60)
print(group_res.size())

그룹별 합계
                      age
survived sex             
0        female   1603.00
         male    11382.50
1        female   5683.00
         male     2536.67

그룹별 데이터 개수 (count - 결측값 제외)
                 age  embark_town
survived sex                     
0        female   64           81
         male    360          468
1        female  197          231
         male     93          109

그룹별 크기 (size - 결측값 포함)
survived  sex   
0         female     81
          male      468
1         female    233
          male      109
dtype: int64


### 6.2 특정 단일 열 대상 집계 연산

In [11]:
# 특정 열(age)에 대해서만 평균 계산
group_res = df.groupby(['survived', 'sex'])
res = group_res['age'].mean()

print("=" * 60)
print("특정 단일 열 대상 집계 연산: age 열의 평균")
print("=" * 60)
print(res)
print(f"\n결과 타입: {type(res)}")

특정 단일 열 대상 집계 연산: age 열의 평균
survived  sex   
0         female    25.046875
          male      31.618056
1         female    28.847716
          male      27.276022
Name: age, dtype: float64

결과 타입: <class 'pandas.core.series.Series'>


### 6.3 산출 결과: Multi-Index Series

- 단일 열 추출 연산의 결과는 **Series** 객체로 반환
- 다차원 데이터에서 1차원 벡터로의 차원 축소 발생
- Multi-Index (다중 인덱스) 구조

In [12]:
# Multi-Index Series 결과 확인
print("=" * 60)
print("Multi-Index Series 구조 확인")
print("=" * 60)
print(f"타입: {type(res)}")
print(f"인덱스 이름: {res.index.names}")
print(f"인덱스 레벨: {res.index.nlevels}")
print(f"\n결과:")
print(res)
print(f"\nName: {res.name}, dtype: {res.dtype}")

Multi-Index Series 구조 확인
타입: <class 'pandas.core.series.Series'>
인덱스 이름: ['survived', 'sex']
인덱스 레벨: 2

결과:
survived  sex   
0         female    25.046875
          male      31.618056
1         female    28.847716
          male      27.276022
Name: age, dtype: float64

Name: age, dtype: float64


In [14]:
# Multi-Index 접근 방법
print("특정 그룹 접근:")
print(f"survived=0, sex='female' 의 평균 나이: {res[(0, 'female')]:.2f}")
print(f"survived=1, sex='male' 의 평균 나이: {res[(1, 'male')]:.2f}")

특정 그룹 접근:
survived=0, sex='female' 의 평균 나이: 25.05
survived=1, sex='male' 의 평균 나이: 27.28


### 6.4 전체 열 대상 집계 연산 적용

In [15]:
# fare 열도 포함하여 분석
df_full = titan.loc[:, ['survived', 'sex', 'age', 'fare']]
group_res = df_full.groupby(['survived', 'sex'])
res = group_res.mean(numeric_only=True)

print("=" * 60)
print("전체 열 대상 집계 연산: mean()")
print("=" * 60)
print(res)
print(f"\n결과 타입: {type(res)}")

전체 열 대상 집계 연산: mean()
                       age       fare
survived sex                         
0        female  25.046875  23.024385
         male    31.618056  21.960993
1        female  28.847716  51.938573
         male    27.276022  40.821484

결과 타입: <class 'pandas.core.frame.DataFrame'>


### 6.5 산출 결과: Multi-Index DataFrame

In [16]:
# Multi-Index DataFrame 구조 확인
print("=" * 60)
print("Multi-Index DataFrame 구조 확인")
print("=" * 60)
print(f"타입: {type(res)}")
print(f"인덱스 이름: {res.index.names}")
print(f"컬럼: {list(res.columns)}")
print(f"Shape: {res.shape}")
print(f"\n결과:")
print(res)

Multi-Index DataFrame 구조 확인
타입: <class 'pandas.core.frame.DataFrame'>
인덱스 이름: ['survived', 'sex']
컬럼: ['age', 'fare']
Shape: (4, 2)

결과:
                       age       fare
survived sex                         
0        female  25.046875  23.024385
         male    31.618056  21.960993
1        female  28.847716  51.938573
         male    27.276022  40.821484


In [17]:
# reset_index()로 일반 DataFrame으로 변환
print("=" * 60)
print("reset_index()로 일반 DataFrame으로 변환")
print("=" * 60)
res_flat = res.reset_index()
print(res_flat)

reset_index()로 일반 DataFrame으로 변환
   survived     sex        age       fare
0         0  female  25.046875  23.024385
1         0    male  31.618056  21.960993
2         1  female  28.847716  51.938573
3         1    male  27.276022  40.821484


---
## 7. Phase 3: Combine (결합) - 고급 매핑 엔진: `agg()` 함수

### 7.1 방법론 1: 사용자 정의 함수 (Custom Function)

In [18]:
# 사용자 정의 함수: 평균 - 최소값 (범위)
def mean_min(x):
    return x.mean() - x.min()

group_res = df_full.groupby(['survived', 'sex'])
result = group_res.agg(mean_min)

print("=" * 60)
print("agg() 방법론 1: 사용자 정의 함수 (mean - min)")
print("=" * 60)
print(result)

agg() 방법론 1: 사용자 정의 함수 (mean - min)
                       age       fare
survived sex                         
0        female  23.046875  16.274385
         male    30.618056  21.960993
1        female  28.097716  44.713573
         male    26.856022  40.821484


In [19]:
# lambda 함수 활용
result_lambda = group_res.agg(lambda x: x.max() - x.min())

print("=" * 60)
print("lambda 함수 활용: 범위(max - min) 계산")
print("=" * 60)
print(result_lambda)

lambda 함수 활용: 범위(max - min) 계산
                   age      fare
survived sex                    
0        female  55.00  144.8000
         male    73.00  263.0000
1        female  62.25  505.1042
         male    79.58  512.3292


### 7.2 방법론 2: 리스트 적용 (List Mapping)

In [20]:
# 리스트 적용: 여러 함수를 동시에 적용
group_res = df_full.groupby(['survived', 'sex'])
mean_min_res = group_res.agg(['mean', 'min'])

print("=" * 60)
print("agg() 방법론 2: 리스트 적용 ['mean', 'min']")
print("=" * 60)
print(mean_min_res)

agg() 방법론 2: 리스트 적용 ['mean', 'min']
                       age             fare       
                      mean   min       mean    min
survived sex                                      
0        female  25.046875  2.00  23.024385  6.750
         male    31.618056  1.00  21.960993  0.000
1        female  28.847716  0.75  51.938573  7.225
         male    27.276022  0.42  40.821484  0.000


### 7.3 산출 결과: Multi-Level Columns

In [21]:
# Multi-Level Columns 구조 확인
print("=" * 60)
print("Multi-Level Columns 구조 확인")
print("=" * 60)
print(f"컬럼 레벨: {mean_min_res.columns.nlevels}")
print(f"컬럼 구조:\n{mean_min_res.columns.tolist()}")
print(f"\n특정 열 접근: age의 mean")
print(mean_min_res[('age', 'mean')])

Multi-Level Columns 구조 확인
컬럼 레벨: 2
컬럼 구조:
[('age', 'mean'), ('age', 'min'), ('fare', 'mean'), ('fare', 'min')]

특정 열 접근: age의 mean
survived  sex   
0         female    25.046875
          male      31.618056
1         female    28.847716
          male      27.276022
Name: (age, mean), dtype: float64


In [25]:
ean_min_res.columns

NameError: name 'ean_min_res' is not defined

In [22]:
# 컬럼 평탄화 (flatten)
mean_min_flat = mean_min_res.copy()
mean_min_flat.columns = ['_'.join(col) for col in mean_min_flat.columns]

print("=" * 60)
print("컬럼 평탄화 후 결과")
print("=" * 60)
print(mean_min_flat.reset_index())

컬럼 평탄화 후 결과
   survived     sex   age_mean  age_min  fare_mean  fare_min
0         0  female  25.046875     2.00  23.024385     6.750
1         0    male  31.618056     1.00  21.960993     0.000
2         1  female  28.847716     0.75  51.938573     7.225
3         1    male  27.276022     0.42  40.821484     0.000


### 7.4 방법론 3: 딕셔너리 매핑 (Dictionary Mapping)


In [26]:
# 딕셔너리 매핑: 열마다 다른 집계 함수 적용
group_res = df_full.groupby(['survived', 'sex'])
result_dict = group_res.agg({'age': 'mean', 'fare': ['mean', 'min']})

print("=" * 60)
print("agg() 방법론 3: 딕셔너리 매핑")
print("  age → mean 만 적용")
print("  fare → mean, min 동시 적용")
print("=" * 60)
print(result_dict)

agg() 방법론 3: 딕셔너리 매핑
  age → mean 만 적용
  fare → mean, min 동시 적용
                       age       fare       
                      mean       mean    min
survived sex                                
0        female  25.046875  23.024385  6.750
         male    31.618056  21.960993  0.000
1        female  28.847716  51.938573  7.225
         male    27.276022  40.821484  0.000


In [27]:
# 사용자 정의 함수 + 딕셔너리 매핑 조합
result_custom = group_res.agg({
    'age': ['mean', 'std', lambda x: x.max() - x.min()],
    'fare': ['sum', 'mean', 'median']
})

print("=" * 60)
print("사용자 정의 함수 + 딕셔너리 매핑 조합")
print("=" * 60)
print(result_custom)

사용자 정의 함수 + 딕셔너리 매핑 조합
                       age                              fare                     
                      mean        std <lambda_0>         sum       mean    median
survived sex                                                                     
0        female  25.046875  13.618591      55.00   1864.9752  23.024385  15.24580
         male    31.618056  14.056019      73.00  10277.7447  21.960993   9.41665
1        female  28.847716  14.175073      62.25  12101.6876  51.938573  26.00000
         male    27.276022  16.504803      79.58   4449.5418  40.821484  26.28750


---
## 8. agg() 파이프라인 매핑 전략 비교 (Synthesis)

| 방법론 | 방식 | Use Case | Output Columns | 단위 매핑 |
|--------|------|----------|----------------|----------|
| 사용자 정의 함수 | `.agg(custom_func)` | 도메인 특화 연산 필요 시 (예: mean-min) | 단일 열 | 단일 |
| 다중 함수 리스트 | `.agg(['mean','min'])` | 동일 필드에 복수 통계량 동시 필요 | 멀티 레벨 | 단일 |
| 딕셔너리 매핑 | `.agg({'col':'func'})` | 필드별 독립적인 연산 설정 필요 | 계층별 구조 | 개별 설정 |

In [28]:
# 전략 비교: 동일한 데이터에 3가지 방법론 적용
group = df_full.groupby(['survived', 'sex'])

print("=" * 60)
print("[전략 1] 사용자 정의 함수")
print("=" * 60)
print(group.agg(lambda x: x.mean() - x.min()))

print("\n" + "=" * 60)
print("[전략 2] 다중 함수 리스트")
print("=" * 60)
print(group.agg(['mean', 'min', 'max']))

print("\n" + "=" * 60)
print("[전략 3] 딕셔너리 매핑")
print("=" * 60)
print(group.agg({'age': 'mean', 'fare': 'sum'}))

[전략 1] 사용자 정의 함수
                       age       fare
survived sex                         
0        female  23.046875  16.274385
         male    30.618056  21.960993
1        female  28.097716  44.713573
         male    26.856022  40.821484

[전략 2] 다중 함수 리스트
                       age                   fare                 
                      mean   min   max       mean    min       max
survived sex                                                      
0        female  25.046875  2.00  57.0  23.024385  6.750  151.5500
         male    31.618056  1.00  74.0  21.960993  0.000  263.0000
1        female  28.847716  0.75  63.0  51.938573  7.225  512.3292
         male    27.276022  0.42  80.0  40.821484  0.000  512.3292

[전략 3] 딕셔너리 매핑
                       age        fare
survived sex                          
0        female  25.046875   1864.9752
         male    31.618056  10277.7447
1        female  28.847716  12101.6876
         male    27.276022   4449.5418
